In [23]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
from pathlib import Path

SUBSETS_DIR = Path("../data/subsets")
OUTPUT_DIR  = Path("../data/subsets")

In [24]:
# Cell 2 — Load audit scores
audit_df = pd.read_csv(SUBSETS_DIR / "label_audit_scores.csv")
print("Audit scores shape:", audit_df.shape)
print(audit_df["dataset"].value_counts())
print("\nColumns:", audit_df.columns.tolist())

Audit scores shape: (17934, 22)
dataset
isic        9158
chexpert    8776
Name: count, dtype: int64

Columns: ['image_id', 'dataset', 'label', 'is_uncertain_original', 'kaggle_path', 'embedding_row', 'extraction_failed', 'knn_score', 'mahal_score', 'iso_score', 'knn_rank', 'mahal_rank', 'iso_rank', 'composite_rank', 'suspicion_score', 'rule_uncertain_label', 'rule_high_knn', 'rule_extreme_mahal', 'rule_all_three_agree', 'rules_fired', 'recommendation', 'loss_weight']


In [ ]:
# Cell 3 — Define suspicion thresholds
# These thresholds are used to determine what action is recommended per image
# Applied independently within each dataset

HIGH_SUSPICION   = 0.80  
MEDIUM_SUSPICION = 0.60  

LOW_SUSPICION    = 0.55  

print("Thresholds:")
print(f"  High   (recommend removal)      : suspicion >= {HIGH_SUSPICION}")
print(f"  Medium (recommend downweighting): suspicion >= {MEDIUM_SUSPICION}")
print(f"  Low    (keep normal)            : suspicion <  {LOW_SUSPICION}")

# Apply per dataset
audit_df["recommendation"] = "keep"

for dataset_name in ["chexpert", "isic"]:
    ds_mask = audit_df["dataset"] == dataset_name

    audit_df.loc[
        ds_mask & (audit_df["suspicion_score"] >= HIGH_SUSPICION),
        "recommendation"
    ] = "remove"

    audit_df.loc[
        ds_mask &
        (audit_df["suspicion_score"] >= MEDIUM_SUSPICION) &
        (audit_df["suspicion_score"] <  HIGH_SUSPICION),
        "recommendation"
    ] = "downweight"

print("\nRecommendation counts per dataset:")
print(audit_df.groupby(["dataset", "recommendation"]).size().to_string())

Thresholds:
  High   (recommend removal)      : suspicion >= 0.8
  Medium (recommend downweighting): suspicion >= 0.6
  Low    (keep normal)            : suspicion <  0.55

Recommendation counts per dataset:
dataset   recommendation
chexpert  downweight        2393
          keep              5974
          remove             409
isic      downweight        2129
          keep              6015
          remove            1014


In [26]:
# Cell 4 — Generate removal list
# Top suspicious images recommended for exclusion from training

removal_list = audit_df[audit_df["recommendation"] == "remove"].copy()
removal_list = removal_list.sort_values(
    ["dataset", "suspicion_score"], ascending=[True, False]
)

print(f"Total images recommended for removal: {len(removal_list)}")
print(f"\nCheXpert removals: "
      f"{(removal_list['dataset']=='chexpert').sum()}")
print(f"ISIC removals:     "
      f"{(removal_list['dataset']=='isic').sum()}")

print("\nTop 10 removal candidates:")
print(removal_list[[
    "image_id", "dataset", "label",
    "suspicion_score", "rules_fired"
]].head(10).to_string())

Total images recommended for removal: 1423

CheXpert removals: 409
ISIC removals:     1014

Top 10 removal candidates:
            image_id   dataset            label  suspicion_score  rules_fired
0  view1_frontal.jpg  chexpert     Lung Opacity         0.961581            0
1  view1_frontal.jpg  chexpert    Pleural Other         0.957289            1
2  view1_frontal.jpg  chexpert         Fracture         0.953832            0
3  view1_frontal.jpg  chexpert      Atelectasis         0.948515            0
4  view1_frontal.jpg  chexpert     Pneumothorax         0.944793            0
5  view1_frontal.jpg  chexpert  Support Devices         0.944679            0
6  view1_frontal.jpg  chexpert      Atelectasis         0.943957            0
7  view1_frontal.jpg  chexpert    Consolidation         0.939931            1
8  view1_frontal.jpg  chexpert            Edema         0.937690            0
9  view1_frontal.jpg  chexpert    Consolidation         0.937424            1


In [ ]:
# Cell 5 — Generate downweight config
# Maps each image to a loss weight for training of classifier
# Suspicious images get lower weight so they contribute less for training

def assign_loss_weight(row):
    if row["recommendation"] == "remove":
        return 0.0    # excluded entirely
    elif row["recommendation"] == "downweight":
        # Scale weight inversely with suspicion score for training
        # suspicion=0.55 → weight≈0.45, suspicion=0.74 → weight≈0.26
        return round(1.0 - row["suspicion_score"], 2)
    else:
        return 1.0    # normal weight

audit_df["loss_weight"] = audit_df.apply(assign_loss_weight, axis=1)

print("Loss weight distribution:")
print(audit_df.groupby("dataset")["loss_weight"].describe().to_string())

print("\nWeight breakdown per dataset:")
for dataset_name in ["chexpert", "isic"]:
    ds = audit_df[audit_df["dataset"] == dataset_name]
    print(f"\n  {dataset_name.upper()}:")
    print(f"    weight = 0.0 (remove)    : "
          f"{(ds['loss_weight'] == 0.0).sum()}")
    print(f"    weight < 1.0 (downweight): "
          f"{((ds['loss_weight'] > 0) & (ds['loss_weight'] < 1)).sum()}")
    print(f"    weight = 1.0 (keep)      : "
          f"{(ds['loss_weight'] == 1.0).sum()}")

Loss weight distribution:
           count      mean       std  min   25%  50%  75%  max
dataset                                                       
chexpert  8776.0  0.766400  0.348043  0.0  0.36  1.0  1.0  1.0
isic      9158.0  0.729079  0.385341  0.0  0.33  1.0  1.0  1.0

Weight breakdown per dataset:

  CHEXPERT:
    weight = 0.0 (remove)    : 409
    weight < 1.0 (downweight): 2393
    weight = 1.0 (keep)      : 5974

  ISIC:
    weight = 0.0 (remove)    : 1014
    weight < 1.0 (downweight): 2129
    weight = 1.0 (keep)      : 6015


In [ ]:
# Cell 6 — Class balance analysis BEFORE cleaning
# Check current class distribution in the full dataset

print("CLASS DISTRIBUTION BEFORE CLEANING\n")

for dataset_name in ["chexpert", "isic"]:
    ds = audit_df[audit_df["dataset"] == dataset_name]
    print(f"{dataset_name.upper()}:")
    counts = ds["label"].value_counts().sort_values(ascending=False)
    for label, count in counts.items():
        print(f"  {label:<35} {count:>4} images")
    print()

CLASS DISTRIBUTION BEFORE CLEANING

CHEXPERT:
  No Finding                          1464 images
  Consolidation                        806 images
  Pneumonia                            743 images
  Edema                                608 images
  Enlarged Cardiomediastinum           580 images
  Fracture                             533 images
  Pneumothorax                         533 images
  Support Devices                      532 images
  Cardiomegaly                         530 images
  Pleural Effusion                     504 images
  Atelectasis                          502 images
  Lung Lesion                          482 images
  Lung Opacity                         480 images
  Pleural Other                        479 images

ISIC:
  BCC                                 2500 images
  MEL                                 2292 images
  NV                                  2235 images
  BKL                                 2131 images



In [ ]:
# Cell 7 — Class balance analysis AFTER cleaning
# Check what happens to class distribution after removals of high suspicion images

print("CLASS DISTRIBUTION AFTER REMOVING HIGH-SUSPICION IMAGES\n")

cleaned_df = audit_df[audit_df["recommendation"] != "remove"].copy()

balance_report = []

for dataset_name in ["chexpert", "isic"]:
    ds_before = audit_df[audit_df["dataset"] == dataset_name]
    ds_after  = cleaned_df[cleaned_df["dataset"] == dataset_name]

    print(f"{dataset_name.upper()}:")
    print(f"  {'Label':<35} {'Before':>7} {'After':>7} {'Removed':>8} {'% Lost':>8}")
    print(f"  {'-'*65}")

    for label in ds_before["label"].unique():
        before = (ds_before["label"] == label).sum()
        after  = (ds_after["label"]  == label).sum()
        removed = before - after
        pct_lost = (removed / before * 100) if before > 0 else 0

        balance_report.append({
            "dataset": dataset_name,
            "label": label,
            "before": before,
            "after": after,
            "removed": removed,
            "pct_lost": round(pct_lost, 1)
        })

        print(f"  {label:<35} {before:>7} {after:>7} {removed:>8} {pct_lost:>7.1f}%")
    print()

balance_df = pd.DataFrame(balance_report)

CLASS DISTRIBUTION AFTER REMOVING HIGH-SUSPICION IMAGES

CHEXPERT:
  Label                                Before   After  Removed   % Lost
  -----------------------------------------------------------------
  Lung Opacity                            480     452       28     5.8%
  Pleural Other                           479     447       32     6.7%
  Fracture                                533     507       26     4.9%
  Atelectasis                             502     443       59    11.8%
  Pneumothorax                            533     510       23     4.3%
  Support Devices                         532     491       41     7.7%
  Consolidation                           806     762       44     5.5%
  Edema                                   608     587       21     3.5%
  Pleural Effusion                        504     487       17     3.4%
  Enlarged Cardiomediastinum              580     549       31     5.3%
  No Finding                             1464    1443       21     1.4%
 

In [ ]:
# Cell 8 — Augmentation recommendations
# Flag any class that loses too many samples after cleaning and recommend augmentation to compensate

MIN_SAMPLES_THRESHOLD = 50   # minimum acceptable class size after cleaning
LOSS_RATE_THRESHOLD   = 0.30  # flag if class loses more than 30% of samples

print("AUGMENTATION RECOMMENDATIONS\n")

needs_augmentation = balance_df[
    (balance_df["after"] < MIN_SAMPLES_THRESHOLD) |
    (balance_df["pct_lost"] > LOSS_RATE_THRESHOLD * 100)
].copy()

if len(needs_augmentation) == 0:
    print("No classes require augmentation — all classes retain "
          "sufficient samples after cleaning.")
else:
    print(f"{'Dataset':<12} {'Label':<35} {'After':>7} "
          f"{'% Lost':>8}  Recommendation")
    print("-" * 80)
    for _, row in needs_augmentation.iterrows():
        target = max(MIN_SAMPLES_THRESHOLD, int(row["before"] * 0.8))
        needed = max(0, target - int(row["after"]))
        print(f"{row['dataset']:<12} {row['label']:<35} "
              f"{int(row['after']):>7} {row['pct_lost']:>7.1f}%  "
              f"Augment by ~{needed} samples")

print("\nSuggested augmentation techniques per dataset:")
print("  CheXpert : horizontal flip, slight rotation (±5°), "
      "contrast adjustment")
print("  ISIC     : horizontal/vertical flip, rotation (±30°), "
      "colour jitter, zoom")

AUGMENTATION RECOMMENDATIONS

No classes require augmentation — all classes retain sufficient samples after cleaning.

Suggested augmentation techniques per dataset:
  CheXpert : horizontal flip, slight rotation (±5°), contrast adjustment
  ISIC     : horizontal/vertical flip, rotation (±30°), colour jitter, zoom


In [31]:
# Cell 9 — Summary statistics for dissertation
print("=" * 60)
print("DATASET LABEL OPTIMIZER — SUMMARY")
print("=" * 60)

for dataset_name in ["chexpert", "isic"]:
    ds = audit_df[audit_df["dataset"] == dataset_name]
    total    = len(ds)
    removed  = (ds["recommendation"] == "remove").sum()
    downwt   = (ds["recommendation"] == "downweight").sum()
    kept     = (ds["recommendation"] == "keep").sum()
    all3     = ds["rule_all_three_agree"].sum()

    print(f"\n{dataset_name.upper()} ({total} images):")
    print(f"  Recommended for removal      : {removed:>4} "
          f"({removed/total*100:.1f}%)")
    print(f"  Recommended for downweighting: {downwt:>4} "
          f"({downwt/total*100:.1f}%)")
    print(f"  Kept at full weight          : {kept:>4} "
          f"({kept/total*100:.1f}%)")
    print(f"  All-three-agree (strongest)  : {all3:>4}")

DATASET LABEL OPTIMIZER — SUMMARY

CHEXPERT (8776 images):
  Recommended for removal      :  409 (4.7%)
  Recommended for downweighting: 2393 (27.3%)
  Kept at full weight          : 5974 (68.1%)
  All-three-agree (strongest)  :  171

ISIC (9158 images):
  Recommended for removal      : 1014 (11.1%)
  Recommended for downweighting: 2129 (23.2%)
  Kept at full weight          : 6015 (65.7%)
  All-three-agree (strongest)  :  416


In [32]:
# Cell 10 — Save all outputs

# 1 — Full audit with recommendations and loss weights
audit_df.to_csv(OUTPUT_DIR / "label_audit_scores.csv", index=False)
print("Updated label_audit_scores.csv with recommendations and loss weights")

# 2 — Removal list only
removal_list.to_csv(OUTPUT_DIR / "removal_list.csv", index=False)
print(f"Saved removal_list.csv ({len(removal_list)} images)")

# 3 — Cleaned dataset manifest (removal candidates excluded)
cleaned_df.to_csv(OUTPUT_DIR / "cleaned_manifest.csv", index=False)
print(f"Saved cleaned_manifest.csv ({len(cleaned_df)} images)")

# 4 — Class balance report
balance_df.to_csv(OUTPUT_DIR / "class_balance_report.csv", index=False)
print(f"Saved class_balance_report.csv")

# 5 — Downweight config (image_id → loss_weight mapping)
downweight_config = audit_df[["image_id", "dataset", "label",
                               "suspicion_score", "loss_weight"]]
downweight_config.to_csv(OUTPUT_DIR / "downweight_config.csv", index=False)
print(f"Saved downweight_config.csv")

Updated label_audit_scores.csv with recommendations and loss weights
Saved removal_list.csv (1423 images)
Saved cleaned_manifest.csv (16511 images)
Saved class_balance_report.csv
Saved downweight_config.csv


In [33]:
# Cell 11 — Final verification
print("FINAL VERIFICATION\n")

removal   = pd.read_csv(OUTPUT_DIR / "removal_list.csv")
cleaned   = pd.read_csv(OUTPUT_DIR / "cleaned_manifest.csv")
balance   = pd.read_csv(OUTPUT_DIR / "class_balance_report.csv")
downwt    = pd.read_csv(OUTPUT_DIR / "downweight_config.csv")

print(f"removal_list.csv     : {len(removal)} images")
print(f"cleaned_manifest.csv : {len(cleaned)} images")
print(f"class_balance_report : {len(balance)} class entries")
print(f"downweight_config    : {len(downwt)} images")
print(f"\nOriginal total : {len(audit_df)}")
print(f"After removal  : {len(cleaned)}")
print(f"Reduction      : {len(audit_df) - len(cleaned)} images "
      f"({(len(audit_df)-len(cleaned))/len(audit_df)*100:.1f}%)")

assert len(removal) + len(cleaned) == len(audit_df), \
    "Removal + cleaned should equal original total"
print("\nAll checks passed. Notebook 08 complete.")

FINAL VERIFICATION

removal_list.csv     : 1423 images
cleaned_manifest.csv : 16511 images
class_balance_report : 18 class entries
downweight_config    : 17934 images

Original total : 17934
After removal  : 16511
Reduction      : 1423 images (7.9%)

All checks passed. Notebook 08 complete.


In [34]:
# Run in Notebook 08 after re-running it
print(audit_df["recommendation"].value_counts())
print("\nRemoval rate:", 
      (audit_df["recommendation"]=="remove").sum() / len(audit_df))

recommendation
keep          11989
downweight     4522
remove         1423
Name: count, dtype: int64

Removal rate: 0.07934649269543884
